In [224]:
import requests as requests
import json
import pandas as pd
import random
import time

In [264]:
keys={}
with open("../keys.json","r") as f:
    keys = json.loads(f.read())
    
# Consumer keys and access tokens, used for OAuth
cds_key = keys["cds_key"]
cds_host = keys["cds_host"]

In [317]:
def sendBatchCdsQuery(total):
    offset = 0
    limit = 50
    entries = []
    while (offset < total):
        batchEntries = pd.DataFrame(sendCdsQuery(offset, limit))
        entries.append(batchEntries)
        if (len(batchEntries) < limit):
            break
            
        offset += 50
        
        random_uniform = random.uniform(3.5, 6.8)
        time.sleep(random_uniform)
    
    return pd.concat(entries)

def sendCdsCollectionQuery(id):
    query_url = cds_host + '/v1/documents/' + str(id)

    
    # Send GET request with bearer token
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {cds_key}'
    }

    response = requests.get(query_url, headers=headers)
    try_count = 0
    entries = {}
    title = ""
    while (response.status_code != 200):
        time.sleep(.5)  
        resp = requests.get(query_url)
            
        if (try_count > 5):
            break
        try_count = try_count + 1
    if (try_count <= 5):        
        result = json.loads(response.text)['resources']
        title = result[0]['title'];
        
    return title

def sendCdsQuery(offset, limit):
    print(offset)
    query_url = cds_host + '/v1/documents' 

    payload = {
        'offset': offset,
        'limit': limit,
        'profileIds': 'story',
        'profileIds': 'buildout',
        'sort': 'publishDateTime:desc',
        'ownerHrefs': 'https://organization.api.npr.org/v4/services/s1'
    }

    # Send GET request with bearer token
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {cds_key}'
    }

    response = requests.get(query_url, params=payload, headers=headers)
    try_count = 0
    entries = {}
    while (response.status_code != 200):
        time.sleep(.5)  
        resp = requests.get(query_url)
            
        if (try_count > 5):
            break
        try_count = try_count + 1
    if (try_count <= 5):        
        entries = json.loads(response.text)['resources']

    return entries[resources][0]

def getTextForStory(story):
    storyText = '';
    for element in story['layout']:
        elementId = element['href'].split('/')[2]
        if elementId in story['assets']:
            asset = story['assets'][elementId]
            assetIsText = False
            for elementProfile in asset['profiles']:
                if (elementProfile['href'] == '/v1/profiles/text'):
                    assetIsText = True
            if assetIsText:
                storyText += (asset['text'] + " ")
    return storyText

def addTextToEntries(entries):
    texts = []
    for index, row in entries.iterrows():
        text = getTextForStory(row)
        texts.append(text)
    entries.insert(1, 'text', texts) # Insert at index 1

    return entries

def getStories(limit):
    entries = sendBatchCdsQuery(limit)
    entries = addTextToEntries(entries)
    trimmedEntries = entries.loc[:, ['id', 'title', 'text', 'profiles', 'publishDateTime', 'collections', 'audio', 'nprWebsitePath', 'meta']]
    trimmedEntries = trimmedEntries.reset_index()

    return trimmedEntries


def addCollectionsToEntries(entries):
    collectionDict = {}
    collectionNames = []
    for index, row in entries.iterrows():
        rowNames = []
        print(index)
        print(len(collectionDict.keys()))
        print("____")
        if (isinstance(row['collections'], list)):
            for collection in row['collections']:
                collectionId = collection['href'].split('/')[3]
                if (collectionId in collectionDict.keys()):
                    collectionName = sendCdsCollectionQuery(collectionId)
                    rowNames.append(collectionDict[collectionId])
                else:
                    collectionName = sendCdsCollectionQuery(collectionId).lower()
                    collectionDict[collectionId] = collectionName;
                    rowNames.append(collectionName)

        collectionNames.append(rowNames)
        
    entries.insert(1, 'collectionNames', collectionNames) # Insert at index 1

    return entries

entries = getStories(5000)

entries.to_csv("stories.csv")  

In [321]:
entries = entries.reset_index()
entriesWithCollections = addCollectionsToEntries(entries)

0
0
____
1
2
____
2
2
____
3
2
____
4
2
____
5
2
____
6
2
____
7
2
____
8
3
____
9
3
____
10
3
____
11
3
____
12
3
____
13
3
____
14
3
____
15
3
____
16
17
____
17
19
____
18
23
____
19
23
____
20
24
____
21
29
____
22
31
____
23
43
____
24
52
____
25
54
____
26
56
____
27
59
____
28
68
____
29
69
____
30
69
____
31
75
____
32
79
____
33
88
____
34
89
____
35
93
____
36
100
____
37
100
____
38
102
____
39
107
____
40
111
____
41
112
____
42
119
____
43
122
____
44
126
____
45
131
____
46
132
____
47
132
____
48
132
____
49
133
____
50
134
____
51
134
____
52
134
____
53
134
____
54
134
____
55
134
____
56
134
____
57
134
____
58
143
____
59
144
____
60
144
____
61
145
____
62
154
____
63
161
____
64
165
____
65
169
____
66
173
____
67
181
____
68
184
____
69
184
____
70
190
____
71
197
____
72
199
____
73
207
____
74
208
____
75
215
____
76
215
____
77
215
____
78
215
____
79
216
____
80
225
____
81
234
____
82
238
____
83
248
____
84
253
____
85
253
____
86
253
____
87
253
____
88
253

639
1062
____
640
1062
____
641
1062
____
642
1063
____
643
1063
____
644
1064
____
645
1064
____
646
1067
____
647
1067
____
648
1067
____
649
1067
____
650
1067
____
651
1067
____
652
1072
____
653
1072
____
654
1073
____
655
1073
____
656
1079
____
657
1080
____
658
1080
____
659
1081
____
660
1084
____
661
1084
____
662
1085
____
663
1086
____
664
1086
____
665
1088
____
666
1088
____
667
1090
____
668
1092
____
669
1094
____
670
1095
____
671
1097
____
672
1098
____
673
1099
____
674
1099
____
675
1099
____
676
1099
____
677
1099
____
678
1099
____
679
1099
____
680
1099
____
681
1099
____
682
1099
____
683
1099
____
684
1103
____
685
1105
____
686
1105
____
687
1106
____
688
1107
____
689
1110
____
690
1110
____
691
1116
____
692
1118
____
693
1119
____
694
1119
____
695
1119
____
696
1119
____
697
1120
____
698
1120
____
699
1120
____
700
1121
____
701
1121
____
702
1121
____
703
1121
____
704
1121
____
705
1121
____
706
1121
____
707
1121
____
708
1122
____
709
1123
____
710
11

1210
1561
____
1211
1562
____
1212
1562
____
1213
1562
____
1214
1562
____
1215
1562
____
1216
1563
____
1217
1565
____
1218
1565
____
1219
1565
____
1220
1565
____
1221
1565
____
1222
1565
____
1223
1568
____
1224
1568
____
1225
1568
____
1226
1568
____
1227
1569
____
1228
1571
____
1229
1571
____
1230
1575
____
1231
1579
____
1232
1582
____
1233
1582
____
1234
1582
____
1235
1582
____
1236
1582
____
1237
1582
____
1238
1582
____
1239
1582
____
1240
1582
____
1241
1582
____
1242
1582
____
1243
1582
____
1244
1582
____
1245
1582
____
1246
1585
____
1247
1585
____
1248
1585
____
1249
1587
____
1250
1587
____
1251
1587
____
1252
1587
____
1253
1587
____
1254
1587
____
1255
1587
____
1256
1590
____
1257
1590
____
1258
1592
____
1259
1594
____
1260
1596
____
1261
1601
____
1262
1601
____
1263
1602
____
1264
1602
____
1265
1605
____
1266
1605
____
1267
1605
____
1268
1605
____
1269
1605
____
1270
1605
____
1271
1605
____
1272
1607
____
1273
1607
____
1274
1607
____
1275
1610
____
1276
1610


1758
1960
____
1759
1963
____
1760
1963
____
1761
1966
____
1762
1966
____
1763
1966
____
1764
1966
____
1765
1966
____
1766
1966
____
1767
1966
____
1768
1966
____
1769
1966
____
1770
1966
____
1771
1966
____
1772
1966
____
1773
1966
____
1774
1966
____
1775
1966
____
1776
1966
____
1777
1969
____
1778
1969
____
1779
1971
____
1780
1974
____
1781
1975
____
1782
1976
____
1783
1977
____
1784
1977
____
1785
1979
____
1786
1979
____
1787
1979
____
1788
1979
____
1789
1979
____
1790
1979
____
1791
1979
____
1792
1980
____
1793
1980
____
1794
1982
____
1795
1982
____
1796
1982
____
1797
1987
____
1798
1987
____
1799
1987
____
1800
1988
____
1801
1988
____
1802
1992
____
1803
1992
____
1804
1992
____
1805
1992
____
1806
1992
____
1807
1994
____
1808
1994
____
1809
1998
____
1810
1998
____
1811
1998
____
1812
2002
____
1813
2002
____
1814
2002
____
1815
2002
____
1816
2002
____
1817
2002
____
1818
2002
____
1819
2002
____
1820
2002
____
1821
2002
____
1822
2002
____
1823
2002
____
1824
2002


In [322]:
entriesWithCollections

,index,collectionNames,id,title,text,profiles,publishDateTime,collections,audio,nprWebsitePath,meta
0,0,"[climate, news]",g-s1-qa722514,Nightwatch Grove for NPR Not A Stream Audio Te...,This is to test if Not A Live Stream Audio is ...,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-11-23T20:15:45.456-05:00,"[{'href': '/v1/documents/1167', 'rels': ['topi...","[{'href': '#/assets/g-s1-qa722516', 'rels': ['...",/2025/11/23/g-s1-qa722514/nightwatch-grove-for...,"{'assetInternalLinkDocuments': [], 'documentLa..."
1,1,"[climate, news]",g-s1-qa722511,Nightwatch Grove for NPR Not A Stream Audio Te...,This is to test if Not A Live Stream Audio is ...,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-11-23T20:00:48.469-05:00,"[{'href': '/v1/documents/1167', 'rels': ['topi...","[{'href': '#/assets/g-s1-qa722513', 'rels': ['...",/2025/11/23/g-s1-qa722511/nightwatch-grove-for...,"{'assetInternalLinkDocuments': [], 'documentLa..."
2,2,"[climate, news]",g-s1-qa722508,Nightwatch Grove for NPR Not A Stream Audio Te...,This is to test if Not A Live Stream Audio is ...,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-11-23T19:54:51.24-05:00,"[{'href': '/v1/documents/1167', 'rels': ['topi...","[{'href': '#/assets/g-s1-qa722510', 'rels': ['...",/2025/11/23/g-s1-qa722508/nightwatch-grove-for...,"{'assetInternalLinkDocuments': [], 'documentLa..."
3,3,[],g-s1-qa722478,Nightwatch Grove for NPR Test Segment: hk388,This is a short body.,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-11-23T18:49:56.612-05:00,NaN,NaN,/2025/11/23/g-s1-qa722478/nightwatch-grove-for...,"{'assetInternalLinkDocuments': [], 'documentLa..."
4,4,"[climate, news]",g-s1-qa722475,Nightwatch Grove for NPR Stream Audio Test: w6osd,This is to test check if stream audio is added...,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-11-23T18:45:02.239-05:00,"[{'href': '/v1/documents/1167', 'rels': ['topi...","[{'href': '#/assets/g-s1-qa722477', 'rels': ['...",/2025/11/23/g-s1-qa722475/nightwatch-grove-for...,"{'assetInternalLinkDocuments': [], 'documentLa..."
...,...,...,...,...,...,...,...,...,...,...,...
1995,45,[fresh air],g-s1-92859,"Fresh Air for Oct. 11, 2025: Dwayne 'The Rock'...",,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-10-11T12:20:31.697-04:00,"[{'href': '/v1/documents/13', 'rels': ['progra...",NaN,/programs/fresh-air/g-s1-92859/fresh-air-for-o...,{'assetInternalLinkDocuments': [{'href': '/v1/...
1996,46,[fresh air weekend],nx-s1-5567849,Fresh Air Weekend: Dwayne 'The Rock' Johnson; ...,Fresh Air Weekend <em>highlights some of the b...,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-10-11T12:17:15.744-04:00,"[{'href': '/v1/documents/139029251', 'rels': [...","[{'href': '#/assets/nx-s1-9488422', 'rels': ['...",/2025/10/11/nx-s1-5567849/fresh-air-weekend-dw...,{'assetInternalLinkDocuments': [{'href': '/v1/...
1997,47,"[africa, world, apple news, news, africa, came...",nx-s1-5570875,Africa's oldest leader isn't ready to retire —...,"JOHANNESBURG, South Africa — After a newspaper...","[{'href': '/v1/profiles/publishable', 'rels': ...",2025-10-11T11:32:10.555-04:00,"[{'href': '/v1/documents/1126', 'rels': ['topi...",NaN,/2025/10/11/nx-s1-5570875/cameroon-biya-africa,"{'assetInternalLinkDocuments': [], 'documentLa..."
1998,48,"[goats and soda, global health, health, news, ...",g-s1-92962,Photos celebrate the glory of girls on 'Intern...,From the slight smile on Yefreannys Isamar Muñ...,"[{'href': '/v1/profiles/publishable', 'rels': ...",2025-10-11T10:21:18.214-04:00,"[{'href': '/v1/documents/327351768', 'rels': [...",NaN,/sections/goats-and-soda/2025/10/11/g-s1-92962...,"{'assetInternalLinkDocuments': [], 'documentLa..."


In [323]:
entriesWithCollections.to_csv("entriesWithCollections.csv")